# Customer spend per day

For the loyalty programme: how many orders each customer placed per day and how much they spent,
from today's raw snapshot of the shop's orders (the same snapshot `orders_daily` is built from).

In [ ]:
from pyspark.sql import functions as F

RAW = "workspace.orders_lab.raw_orders"
TARGET = "workspace.orders_lab.customer_daily"

In [ ]:
raw = spark.table(RAW)
latest = raw.agg(F.max("dt")).first()[0]
# loyalty points are for real purchases only
snapshot = raw.where(F.col("dt") == F.lit(latest)).where(F.col("status") != "CANCELLED")

In [ ]:
customer_daily = (
    snapshot
    .groupBy("customer_id", F.to_date("created_at").alias("order_date"))
    .agg(
        F.count(F.lit(1)).alias("order_count"),
        F.sum("amount").cast("decimal(22,2)").alias("total_amount"),
    )
)

In [ ]:
(customer_daily.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TARGET))
print(f"{TARGET}: {customer_daily.count()} rows from the snapshot of {latest}")